# Ythan Estuary Food Web — Network Analysis (Group Assignment)

This notebook reproduces **our part** of the group assignment (last two points):
1. **Node centralities** (≥3 measures, including a spectral measure; in/out variants for directed measures)
2. **Extra tool**: multiple **community detection** algorithms + **trophic-level** diagnostic

**Edge convention (important):** `prey → predator` (an edge A→B means A is eaten by B).

**How to use with your group**
- Run cells top-to-bottom in **conda (base)** (or any env with `networkx`, `pandas`, `matplotlib`, `numpy`).
- The first code cell can **clone the GitHub repo** (Python code + `data/Ythan.txt` only).
- All heavy lifting is in the `analysis/` Python modules; this notebook calls those functions and shows plots inline.

In [ ]:
# --- Optional: clone the repository (for group members who only received this notebook) ---
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/joelinator/NetworkTheoryAssignment.git"  # update if repo name differs
CLONE_DIR = Path.cwd() / "NetworkTheoryAssignment"

if not (Path.cwd() / "analysis").exists() and not CLONE_DIR.exists():
    print(f"Cloning {REPO_URL} ...")
    subprocess.run(["git", "clone", REPO_URL, str(CLONE_DIR)], check=True)
    os.chdir(CLONE_DIR)
    print("Cloned into:", CLONE_DIR)
elif CLONE_DIR.exists() and not (Path.cwd() / "analysis").exists():
    os.chdir(CLONE_DIR)
    print("Using existing clone:", CLONE_DIR.resolve())
else:
    print("Working directory:", Path.cwd().resolve())

In [ ]:
# --- Setup: imports + paths ---
import sys
from pathlib import Path
from IPython.display import display, Image, Markdown

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "analysis").exists():
    raise FileNotFoundError("Cannot find analysis/ folder. Run the clone cell above or open the notebook from the repo root.")

sys.path.insert(0, str(ROOT / "analysis"))

from load_network import load_ythan, giant_weakly_connected_subgraph
from centrality_analysis import compute_centralities, save_centrality_outputs, centrality_summary_text
from extra_tool_analysis import save_extra_tool_outputs, detect_communities_undirected
from visualize_network import compute_layout, plot_network_by_centrality, plot_network_by_communities

DATA = ROOT / "data" / "Ythan.txt"
OUT = ROOT / "outputs"
OUT.mkdir(parents=True, exist_ok=True)

%matplotlib inline
plt.rcParams["figure.dpi"] = 120

print("ROOT:", ROOT.resolve())
print("Dataset:", DATA)
print("Outputs:", OUT.resolve())

## 1) Load the directed food web

**Ecology terms**
- **Food web**: who-eats-whom network.
- **Prey → predator**: arrows follow energy flow upward.

**Network terms**
- **Directed graph**: edges have a direction.
- **Weakly connected component (WCC)**: connected if you ignore direction; we use the giant WCC when needed.

In [ ]:
nd = load_ythan(DATA)
G = nd.G
G_wcc = giant_weakly_connected_subgraph(G)

print(f"Nodes: {G.number_of_nodes()}, Edges: {G.number_of_edges()}")
print(f"Giant WCC: {G_wcc.number_of_nodes()} nodes, {G_wcc.number_of_edges()} edges")

# Preview edge list
nd.edges.head()

## 2) Centrality measures (directed: in vs out)

We compute several centralities on the **directed** graph:

| Measure | Meaning in this food web (prey→predator) |
|---------|------------------------------------------|
| **in_degree** | Diet breadth / **generality** (how many prey a predator has) |
| **out_degree** | **Vulnerability** (how many predators eat this species) |
| **betweenness** | **Connector** on shortest directed paths |
| **closeness_in / closeness_out** | Reachability *to* vs *from* a node (via \(G\) vs \(G^R\)) |
| **pagerank_in / pagerank_out** | Spectral importance on \(G\) vs reversed \(G^R\) |

**Interpretation tip:** predator-side measures (`in_degree`, `pagerank_in`, `closeness_out`) often highlight generalist predators; resource-side measures (`out_degree`, `pagerank_out`, `closeness_in`) highlight basal/widely consumed species.

In [ ]:
cent = compute_centralities(G_wcc)
save_centrality_outputs(cent, OUT, prefix="ythan")

print(centrality_summary_text(cent))

# Show full table (sortable)
cent_df = cent.centralities.reset_index()
display(cent_df.sort_values("pagerank_in", ascending=False).head(10))

In [ ]:
# Correlation heatmap (saved to outputs/)
from IPython.display import Image
Image(filename=OUT / "ythan_centrality_corr_heatmap.png")

### Results snapshot (what we found)

From our run on this network:
- **Top in-degree (generalists):** 118, 122, 124
- **Top out-degree (vulnerable resources):** 3, 5, 19
- **Top pagerank_in (predator/sink importance):** 132, 124, 118 — node **132** has out-degree 0 → consistent with a **top predator / sink**
- **Top pagerank_out (resource importance):** 5, 19, 0 — aligns with **basal / widely consumed** species

**Spearman correlations:** `pagerank_in` ≈ `in_degree` (ρ≈0.98) and `pagerank_out` ≈ `out_degree` (ρ≈0.91) → two ecological “axes”: predator importance vs resource importance.

## 3) Network visualisations (directed arrows + centrality gradients)

Nodes are **colored and sized** by centrality; edges show **arrow direction** (prey→predator).

In [ ]:
pos = compute_layout(G_wcc, seed=7)
cdf = cent.centralities.reset_index().rename(columns={"index": "node"})

plots = [
    ("in_degree", "viridis"),
    ("out_degree", "magma"),
    ("betweenness", "plasma"),
    ("pagerank_in", "cividis"),
    ("pagerank_out", "cividis"),
]

for col, cmap in plots:
    values = {int(r["node"]): float(r[col]) for _, r in cdf.iterrows()}
    out_path = OUT / f"ythan_network_{col}.png"
    plot_network_by_centrality(
        G_wcc, pos, values,
        out_path=out_path,
        title=f"Ythan — {col} (size+color), arrows = prey→predator",
        cmap=cmap,
    )
    display(Markdown(f"### {col}"))
    display(Image(filename=out_path))

## 4) Community detection (multiple algorithms)

**Network theory**
- A **community** (module) has denser internal links than expected by chance.
- **Modularity \(Q\)** scores how strong that compartmentalization is (\(Q\approx 0.3\) is often meaningful in empirical networks).

**Ecology**
- Communities may reflect **habitat compartments**, **guilds**, or **energy channels** (e.g. detrital vs grazing pathways).

We compare: **Louvain**, **greedy modularity**, **label propagation**, **fluid communities**, **Girvan–Newman** (on an undirected projection of the food web).

In [ ]:
comm_best, troph = save_extra_tool_outputs(G_wcc, OUT, prefix="ythan")

summary = pd.read_csv(OUT / "ythan_communities_methods_summary.csv")
display(summary.sort_values("modularity_Q", ascending=False))

print(f"\nBest partition: {comm_best.method}, k={len(comm_best.communities)}, Q={comm_best.modularity:.3f}")

In [ ]:
# Community-colored directed network (best Q partition)
comm_plot = OUT / f"ythan_network_communities_{comm_best.method}.png"
plot_network_by_communities(
    G_wcc, pos,
    membership=comm_best.membership,
    out_path=comm_plot,
    title=f"Ythan — communities ({comm_best.method})",
)
display(Image(filename=comm_plot))

### Community results (interpretation)

Typical results on this network:
- **Louvain** gives the highest \(Q\) (~0.31) with **5 communities** → non-trivial mesoscale structure.
- **Greedy modularity** is similar (\(Q\approx 0.31\), 4 communities) → structure is not an artifact of one algorithm.
- **Label propagation** and **Girvan–Newman (first split)** can collapse to very coarse 2-way partitions on some graphs.
- **Fluid (k=12)** tends to **over-split** (many small communities, lower \(Q\)).

Ecologically: modular structure supports **compartmentalization** (subwebs that interact more internally), even without species names on nodes.

## 5) Trophic-level diagnostic

**Basal species** (no prey): in-degree = 0 in our convention.

**Trophic level (TL)** estimated by:
\[
TL_i = 1 + \text{mean}(TL_{\text{prey of }i})
\]
(basal nodes have TL = 1)

This is a simple descriptive tool (not a full diet-fraction model), but it checks the expected **pyramid**: many low-TL nodes, fewer high-TL nodes.

In [ ]:
print(f"Basal nodes (in-degree=0): {len(troph.basal_nodes)}")

display(troph.node_table.sort_values("trophic_level", ascending=False).head(10))

display(Image(filename=OUT / "ythan_trophic_level_hist.png"))

## 6) Takeaways for the presentation (my part)

1. **Centrality is role-dependent** in directed food webs: predator measures (in-degree, pagerank_in) ≠ resource measures (out-degree, pagerank_out).
2. **Node 132** is a strong **top-predator/sink** signal (high pagerank_in, out-degree 0).
3. **Nodes 3, 5, 19** are strong **resource/basal** signals (high out-degree, high pagerank_out).
4. **Communities** exist (Louvain \(Q\approx 0.31\), 5 modules) → compartmentalization.
5. **Trophic levels** show the expected layered structure (many basal/low TL).

All CSV/plots are saved under `outputs/` for slides.